# Dataset de entrenamiento

Una fila por merchant. El modelo ordena a los activos: ¿cuáles se parecen a los que una revisión termina suspendiendo?

Decisiones de `docs/decisions.md` (punto 1):

- **Positivo:** merchant con inicio observado (un día activo seguido de uno suspendido). La fila es su primer día suspendido. Esa fila resume la actividad hasta el día anterior, cuando todavía operaba.
- **`M002D3A13420843D9` queda fuera**, también de la sensibilidad: su historia empieza el 1-abr, primer día de datos.
- **`ME9BD50FCE697D4A7`, `MF000975FCCCF8010`, `M315DBFD11714A6D5`, `M486B452BBACA6CA4`, `M3182C56E1D13EC33` y `MEA1EE5CA0A25553F` quedan fuera del análisis.** No entran a entrenamiento ni a test. `MEA1EE5` nunca operó. `M3182` no hizo nada raro.
- **Negativo:** merchant que no fue suspendido en abril–septiembre. Un día activo, con la fecha sorteada como las de los positivos (semilla 0).
- **Fuera:** lo posterior al primer día suspendido, los 7 sin inicio observado, los días con estado nulo, octubre–diciembre, y el resto de la historia.

Salida: `data/processed/dataset.parquet` con `merchant_id`, `observation_date`, `target`, `group_d` y las columnas originales.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
MERCHANT_PATH = ROOT / "data" / "merchant_rating_featureset_sample.parquet"
OUT_PATH = ROOT / "data" / "processed" / "dataset.parquet"

LABEL_END = pd.Timestamp("2025-09-30")
SEED = 0

# Historia observable desde el primer día de datos: no entra ni en la sensibilidad.
LEFT_CENSORED = ["M002D3A13420843D9"]

# No cuentan como casos. Fuera de entrenamiento y de test.
OUT_OF_ANALYSIS = [
    "ME9BD50FCE697D4A7", "MF000975FCCCF8010", "M315DBFD11714A6D5",
    "M486B452BBACA6CA4", "M3182C56E1D13EC33", "MEA1EE5CA0A25553F",
]

# M002 ya está en LEFT_CENSORED. group_d queda vacío: MEA1 nunca operó y no entra.
GROUP_D = [
    "M002D3A13420843D9",
]

## Inicio de cada suspensión

Inicio = primer día suspendido cuyo día anterior en los datos estaba activo. Se calcula sobre abril–septiembre, con las filas ordenadas, antes de quitar los nulos: un nulo no cuenta como activo.

In [ ]:
state_cols = ["merchant_id", "observation_date", "is_suspended"]
data = pd.read_parquet(MERCHANT_PATH, columns=state_cols, filters=[("observation_date", "<=", LABEL_END.date())])
data["observation_date"] = pd.to_datetime(data["observation_date"])
data = data.sort_values(["merchant_id", "observation_date"]).reset_index(drop=True)

state = data["is_suspended"]
previous = state.groupby(data["merchant_id"]).shift(1)
starts = data.loc[(state == True) & (previous == False), ["merchant_id", "observation_date"]]  # noqa: E712
onset = starts.groupby("merchant_id")["observation_date"].min().rename("onset")

ever_suspended = set(data.loc[state == True, "merchant_id"])  # noqa: E712
without_onset = ever_suspended - set(onset.index)
positives = onset.drop(index=LEFT_CENSORED + OUT_OF_ANALYSIS)

print(f"alguna vez suspendidos: {len(ever_suspended)}")
print(f"con inicio observado:   {len(onset)}")
print(f"sin inicio observado:   {len(without_onset)} (se excluyen)")
print(f"censurado al inicio:    {len(LEFT_CENSORED)} (se excluye)")
print(f"fuera del análisis:     {len(OUT_OF_ANALYSIS)} (se excluyen)")
print(f"positivos:              {len(positives)}")

## Qué día se mira

Del positivo, su primer día suspendido. Del negativo, un día activo en alguna de esas fechas, sorteado con el mismo peso que tienen en los positivos. Quien no tiene fila activa en ninguna de esas fechas queda fuera.

In [ ]:
active = data.loc[data["is_suspended"] == False, ["merchant_id", "observation_date"]]  # noqa: E712

# Positivo: la fila de su primer día suspendido. Resume la actividad hasta el día anterior,
# cuando el merchant todavía operaba (un suspendido no genera transacciones).
snapshot = positives.rename("observation_date").reset_index().merge(
    data[["merchant_id", "observation_date", "is_suspended"]], on=["merchant_id", "observation_date"], how="left"
)
assert (snapshot["is_suspended"] == True).all(), "la foto no es un día suspendido"  # noqa: E712
snapshot = snapshot[["merchant_id", "observation_date"]]

pos_keys = snapshot[["merchant_id", "observation_date"]].copy()
pos_keys["target"] = np.int8(1)
pos_keys["group_d"] = pos_keys["merchant_id"].isin(GROUP_D)

weights = pos_keys["observation_date"].value_counts(normalize=True)
candidates = active.loc[
    ~active["merchant_id"].isin(ever_suspended) & active["observation_date"].isin(weights.index)
].copy()
never = set(data["merchant_id"]) - ever_suspended
rng = np.random.default_rng(SEED)
candidates["key"] = np.log(candidates["observation_date"].map(weights).to_numpy()) + rng.gumbel(size=len(candidates))
neg_keys = candidates.loc[candidates.groupby("merchant_id")["key"].idxmax(), ["merchant_id", "observation_date"]].copy()
neg_keys["target"] = np.int8(0)
neg_keys["group_d"] = False

keys = pd.concat([pos_keys, neg_keys], ignore_index=True)
assert keys["merchant_id"].is_unique
assert not set(keys["merchant_id"]) & set(LEFT_CENSORED + OUT_OF_ANALYSIS)
assert not set(keys["merchant_id"]) & without_onset

print(f"positivos: {len(pos_keys)} (group_d: {int(pos_keys['group_d'].sum())})")
print(f"negativos: {len(neg_keys)} (sin fila en esas fechas: {len(never) - len(neg_keys):,})")

mix = pd.DataFrame({
    "positivos": pos_keys["observation_date"].value_counts(normalize=True),
    "negativos": neg_keys["observation_date"].value_counts(normalize=True),
}).sort_index().fillna(0)
print((mix * 100).round(1).to_string())

## Armar el dataset

Se leen solo las fechas de las fotos. En la fila del positivo `is_suspended` figura suspendido (es su primer día suspendido) y en la del negativo, activo. La columna se saca igual.

In [ ]:
dates = [d.date() for d in sorted(keys["observation_date"].unique())]
rows = pd.read_parquet(MERCHANT_PATH, filters=[("observation_date", "in", dates)])
rows["observation_date"] = pd.to_datetime(rows["observation_date"])

dataset = keys.merge(rows, on=["merchant_id", "observation_date"], how="left", validate="one_to_one")
assert dataset["is_suspended"].notna().all()
assert (dataset.loc[dataset["target"] == 1, "is_suspended"] == True).all()  # noqa: E712
assert (dataset.loc[dataset["target"] == 0, "is_suspended"] == False).all()  # noqa: E712
dataset = dataset.drop(columns=["is_suspended"])

assert dataset["merchant_id"].is_unique
assert int(dataset["target"].sum()) == len(positives)
assert set(dataset.loc[dataset["group_d"], "merchant_id"]) == set(GROUP_D) - set(LEFT_CENSORED)

print(f"{len(dataset):,} filas, {dataset['merchant_id'].nunique():,} merchants")
print(f"positivos: {int(dataset['target'].sum())} ({dataset['target'].mean():.3%})")
print(f"group_d: {int(dataset['group_d'].sum())}")

OUT_PATH.parent.mkdir(exist_ok=True)
dataset.to_parquet(OUT_PATH, index=False)
print("guardado", OUT_PATH)